# 02 Embeddings And Pairs
Embeddings erzeugen/laden, QC, LSPO/ADS Pair-Building.

In [1]:
from pathlib import Path
import sys

RUN_STAGE = "smoke"  # smoke|mini|mid|full
PATHS_CONFIG = "configs/paths.local.yaml"  # alternativ: configs/paths.colab.yaml

def _find_root(start: Path) -> Path:
    for c in [start, *start.parents]:
        if (c / "src").exists() and (c / "configs").exists():
            return c.resolve()
    return start.resolve()

ROOT = _find_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("ROOT:", ROOT)

RUN_ID = None  # Optional override. Default: auto from notebook 00 latest_run.json
USE_STUB_EMBEDDINGS = False
PREFER_PRECOMPUTED_ADS = True
DEVICE = "auto"


ROOT: C:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\Author_Name_Disambiguation


In [2]:
import json
import numpy as np
import pandas as pd

from src.common.config import (
    load_notebook_context,
    build_run_dirs,
    resolve_run_id,
    write_run_consistency,
)
from src.common.io_schema import read_parquet, save_parquet
from src.features.embed_chars2vec import get_or_create_chars2vec_embeddings
from src.features.embed_specter import get_or_create_specter_embeddings
from src.approaches.nand.build_pairs import assign_lspo_splits, build_pairs_within_blocks, write_pairs

CTX = load_notebook_context(run_stage=RUN_STAGE, project_root=ROOT, paths_config=PATHS_CONFIG)
DATA = CTX["DATA"]
ART = CTX["ART"]
MODEL = CTX["MODEL"]
RUN = CTX["RUN"]

latest_context_path = Path(ART["metrics_dir"]) / "latest_run.json"
RUN_ID = resolve_run_id(RUN_ID, latest_context_path, allow_placeholder=False)
RUN_DIRS = build_run_dirs(DATA, ART, RUN_ID)
for key in ["subset_cache", "embeddings", "metrics"]:
    RUN_DIRS[key].mkdir(parents=True, exist_ok=True)

write_run_consistency(
    run_id=RUN_ID,
    run_stage=RUN_STAGE,
    run_dirs=RUN_DIRS,
    output_path=RUN_DIRS["metrics"] / "02_run_consistency.json",
    extras={"notebook": "02_embeddings_and_pairs", "latest_context_path": str(latest_context_path)},
)

subset_dir = RUN_DIRS["subset_cache"]
emb_dir = RUN_DIRS["embeddings"]
metrics_dir = RUN_DIRS["metrics"]

lspo_mentions = read_parquet(subset_dir / f"lspo_mentions_{RUN_STAGE}.parquet")
ads_mentions = read_parquet(subset_dir / f"ads_mentions_{RUN_STAGE}.parquet")

print("RUN_ID:", RUN_ID)


RUN_ID: smoke_20260212T200818Z_6b207060


In [3]:
rep = MODEL["representation"]

lspo_chars = get_or_create_chars2vec_embeddings(
    mentions=lspo_mentions,
    output_path=emb_dir / f"lspo_chars2vec_{RUN_STAGE}.npy",
    use_stub_if_missing=USE_STUB_EMBEDDINGS,
)
lspo_text = get_or_create_specter_embeddings(
    mentions=lspo_mentions,
    output_path=emb_dir / f"lspo_specter_{RUN_STAGE}.npy",
    model_name=rep.get("text_model_name", "allenai/specter"),
    max_length=int(rep.get("max_length", 256)),
    batch_size=16,
    device=DEVICE,
    prefer_precomputed=False,
    use_stub_if_missing=USE_STUB_EMBEDDINGS,
)

ads_chars = get_or_create_chars2vec_embeddings(
    mentions=ads_mentions,
    output_path=emb_dir / f"ads_chars2vec_{RUN_STAGE}.npy",
    use_stub_if_missing=USE_STUB_EMBEDDINGS,
)
ads_text = get_or_create_specter_embeddings(
    mentions=ads_mentions,
    output_path=emb_dir / f"ads_specter_{RUN_STAGE}.npy",
    model_name=rep.get("text_model_name", "allenai/specter"),
    max_length=int(rep.get("max_length", 256)),
    batch_size=32,
    device=DEVICE,
    prefer_precomputed=PREFER_PRECOMPUTED_ADS,
    use_stub_if_missing=USE_STUB_EMBEDDINGS,
)

print("lspo chars/text:", lspo_chars.shape, lspo_text.shape)
print("ads chars/text:", ads_chars.shape, ads_text.shape)


lspo chars/text: (1000, 50) (1000, 768)
ads chars/text: (1000, 50) (1000, 768)


In [4]:
def emb_qc(name, arr):
    norms = np.linalg.norm(arr, axis=1)
    return {
        "name": name,
        "shape": tuple(arr.shape),
        "nan_count": int(np.isnan(arr).sum()),
        "norm_mean": float(np.mean(norms)),
        "norm_p01": float(np.quantile(norms, 0.01)),
        "norm_p99": float(np.quantile(norms, 0.99)),
    }

pd.DataFrame([
    emb_qc("lspo_chars", lspo_chars),
    emb_qc("lspo_text", lspo_text),
    emb_qc("ads_chars", ads_chars),
    emb_qc("ads_text", ads_text),
])


,name,shape,nan_count,norm_mean,norm_p01,norm_p99
0,lspo_chars,"(1000, 50)",0,3.672508,2.798476,4.424648
1,lspo_text,"(1000, 768)",0,22.548742,21.580319,23.324893
2,ads_chars,"(1000, 50)",0,3.264804,2.289374,4.071147
3,ads_text,"(1000, 768)",0,22.324959,20.992240,23.181394


In [5]:
split_balance_cfg = RUN.get("split_balance", {})
lspo_mentions, split_meta = assign_lspo_splits(
    lspo_mentions,
    seed=int(RUN.get("seed", 11)),
    min_neg_val=int(split_balance_cfg.get("min_neg_val", 0)),
    min_neg_test=int(split_balance_cfg.get("min_neg_test", 0)),
    max_attempts=int(split_balance_cfg.get("max_attempts", 1)),
    return_meta=True,
)

lspo_pairs = build_pairs_within_blocks(
    mentions=lspo_mentions,
    max_pairs_per_block=RUN.get("max_pairs_per_block"),
    seed=int(RUN.get("seed", 11)),
    require_same_split=True,
    labeled_only=False,
    balance_train=True,
)

lspo_pairs_path = subset_dir / f"lspo_pairs_{RUN_STAGE}.parquet"
write_pairs(lspo_pairs, lspo_pairs_path)
save_parquet(lspo_mentions, subset_dir / f"lspo_mentions_{RUN_STAGE}.parquet", index=False)

with (metrics_dir / "02_split_balance.json").open("w", encoding="utf-8") as f:
    json.dump(split_meta, f, indent=2)

print("Split balancing:", split_meta)
print("LSPO pairs:", len(lspo_pairs), "->", lspo_pairs_path)


Split balancing: {'status': 'split_balance_degraded', 'attempts': 200, 'min_neg_val': 10, 'min_neg_test': 10, 'split_label_counts': {'train': {'pos': 310, 'neg': 75, 'labeled_pairs': 385}, 'val': {'pos': 35, 'neg': 4, 'labeled_pairs': 39}, 'test': {'pos': 33, 'neg': 3, 'labeled_pairs': 36}}}
LSPO pairs: 460 -> C:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\Author_Name_Disambiguation\data\subsets\cache\smoke_20260212T200818Z_6b207060\lspo_pairs_smoke.parquet


In [6]:
ads_pairs = build_pairs_within_blocks(
    mentions=ads_mentions,
    max_pairs_per_block=RUN.get("max_pairs_per_block"),
    seed=int(RUN.get("seed", 11)),
    require_same_split=False,
    labeled_only=False,
    balance_train=False,
)
ads_pairs_path = subset_dir / f"ads_pairs_{RUN_STAGE}.parquet"
write_pairs(ads_pairs, ads_pairs_path)
print("ADS pairs:", len(ads_pairs), "->", ads_pairs_path)


ADS pairs: 500 -> C:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\Author_Name_Disambiguation\data\subsets\cache\smoke_20260212T200818Z_6b207060\ads_pairs_smoke.parquet


In [7]:
def summarize_split_labels(pairs: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for split in ["train", "val", "test", "inference", "mixed"]:
        sub = pairs[pairs["split"] == split]
        known = sub[sub["label"].notna()] if "label" in sub.columns else sub.iloc[0:0]
        rows.append({
            "split": split,
            "pairs": int(len(sub)),
            "labeled_pairs": int(len(known)),
            "pos": int((known["label"] == 1).sum()) if len(known) else 0,
            "neg": int((known["label"] == 0).sum()) if len(known) else 0,
        })
    return pd.DataFrame(rows)

split_counts = summarize_split_labels(lspo_pairs)
print("LSPO split label counts")
display(split_counts)

q = lspo_pairs.copy()
if "label" in q.columns:
    known = q[q["label"].notna()]
    pos = int((known["label"] == 1).sum())
    neg = int((known["label"] == 0).sum())
    ratio = pos / max(1, neg)
    print({"labeled_pairs": len(known), "pos": pos, "neg": neg, "pos_neg_ratio": ratio})

print("Pairs per block:")
print(q.groupby("block_key").size().describe())

leak = 0
if "orcid" in lspo_mentions.columns and "split" in lspo_mentions.columns:
    g = lspo_mentions[lspo_mentions["orcid"].notna()].groupby("orcid")["split"].nunique()
    leak = int((g > 1).sum())
print("orcid leakage groups:", leak)

split_label_counts = json.loads(split_counts.to_json(orient="records"))

with (metrics_dir / "02_pairs_qc.json").open("w", encoding="utf-8") as f:
    json.dump(
        {
            "orcid_leakage_groups": leak,
            "lspo_pairs": int(len(lspo_pairs)),
            "ads_pairs": int(len(ads_pairs)),
            "split_label_counts": split_label_counts,
            "split_balance": split_meta,
        },
        f,
        indent=2,
    )


LSPO split label counts


,split,pairs,labeled_pairs,pos,neg
0,train,385,385,310,75
1,val,39,39,35,4
2,test,36,36,33,3
3,inference,0,0,0,0
4,mixed,0,0,0,0


{'labeled_pairs': 460, 'pos': 378, 'neg': 82, 'pos_neg_ratio': 4.609756097560975}
Pairs per block:
count    460.0
mean       1.0
std        0.0
min        1.0
25%        1.0
50%        1.0
75%        1.0
max        1.0
dtype: float64
orcid leakage groups: 0
